# Setting up an OpenMM iMD simulation

This notebook demonstrates how to set up an OpenMM simulation for use with NanoVer from scratch.
We take AMBER files for neuraminidase with oseltamivir (AKA tamiflu) bound, create an OpenMM system and
set it up with NanoVer.

## OpenMM simulation setup

We start by creating an OpenMM simulation for the neuraminidase-oseltamivir system using the relevant pre-prepared AMBER files. This setup is adapted from the [OpenMM documentation](https://docs.openmm.org/latest/userguide/application/02_running_sims.html#using-amber-files) which also includes examples for Gromacs and CHARMM which could be used instead.

In [1]:
from openmm import unit, LangevinIntegrator
from openmm.app import Simulation, AmberInpcrdFile, AmberPrmtopFile, CutoffPeriodic, OBC2

inpcrd = AmberInpcrdFile("../systems/3TI6_ose_wt.rst")
prmtop = AmberPrmtopFile("../systems/3TI6_ose_wt.top", periodicBoxVectors=inpcrd.boxVectors)

system = prmtop.createSystem(
    nonbondedMethod=CutoffPeriodic,
    nonbondedCutoff=2*unit.nanometer,
    implicitSolvent=OBC2,
    constraints=None,
)

integrator = LangevinIntegrator(
    300*unit.kelvin,
    1/unit.picosecond,
    0.001*unit.picoseconds,
)

simulation = Simulation(prmtop.topology, system, integrator)
simulation.context.setPositions(inpcrd.positions)

C:\Users\ragzo\Documents\REPOS\nanover-server-py-uv\.venv\Lib\site-packages\openmm\app\internal\amber_file_parser.py:1168: UserWarning: Non-optimal GB parameters detected for GB model OBC2
  warnings.warn(


In [2]:
# NBVAL_SKIP
simulation.minimizeEnergy()

As in the OpenMM documentation, we'll run a few steps to check it's stable; printing the potential energy and temperature every 500 steps:

In [3]:
# NBVAL_SKIP
from sys import stdout
from openmm.app import StateDataReporter

simulation.reporters.append(StateDataReporter(stdout, 500, step=True, potentialEnergy=True, temperature=True))
simulation.step(5000)

#"Step","Potential Energy (kJ/mole)","Temperature (K)"
500,-42702.62326834821,121.36254062614057
1000,-37475.11997244977,191.80505551165876
1500,-34311.15073416852,234.11845951404868
2000,-32400.843666297427,256.95612400988745
2500,-31331.96701834821,271.74163612484045
3000,-30376.24204276227,282.8426640649133
3500,-30289.198707801334,287.5676540081011
4000,-30386.710548621646,297.733582629399
4500,-30304.976722938052,292.5954644006916
5000,-29890.976539832584,294.9337188190697


Looks good! Now, let's remove that reporter and set it up for use with NanoVer.

In [4]:
simulation.reporters.clear()

## Saving the OpenMM simulation setup (optional)

The following cell outputs the simulation as a NanoVer OpenMM bundle, which contains the PDB with the topology and OpenMM serialized System and Integrator. This lets you take the simulation we've set up here and use it directly with NanoVer without having to repeat this procedure each time, perfect if you just want to run a simulation quickly. Here's how you would run this simulation from the file created using the terminal:

```bash 
$ nanover-server --omm neuraminidase_nanover.openmm.zip
```

In [5]:
from nanover.openmm.bundles import bundle_openmm_simulation

bundle_openmm_simulation(simulation, outfile="neuraminidase_nanover.openmm.zip")

## NanoVer simulation and server setup

Next we wrap the universe in and host it with NanoVer in the usual way:

In [6]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

omm_sim = OpenMMSimulation.from_simulation(simulation)

imd_runner = OmniRunner.with_basic_server(omm_sim, port=0, name="neuraminidase example")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "neuraminidase example" (ws://localhost:61956), discoverable on all interfaces on port 54545
Available simulations:
[0]: "Unnamed OpenMM Simulation"
Switched to [0]: "Unnamed OpenMM Simulation"
Switched to [0]: "Unnamed OpenMM Simulation"


Great! Now we have our simulation up and running! Connect to it from VR and you'll see something like this:

![NanoVer neuraminidase](../figures/neuraminidase_ball_and_stick.png)

Let's leave it running in the background and turn our attention to an important aspect of molecular visualisation: making things look pretty!

## Let's make it pretty!

Ball and stick is so 2001, let's make it look cool. We'll also make it so that if you interact with any of the atoms of oseltamivir, you'll interact with the entire molecule as a group, which is more stable.

We'll use utility functions for modifying selections.

In [7]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

We'll also define a function that copies color gradients from matplotlib:

In [8]:
import matplotlib

def get_matplotlib_gradient(name: str):
    cmap = matplotlib.colormaps[name]
    return list(list(cmap(x/7)) for x in range(0, 8, 1))

Now we'll get started by hiding the `root` selection, which always contains every atom of the system. Then we'll introduce new selections for each individual part of the system that we want to see.

In [9]:
utilities.selections.update_selection("root", hide=True)

We'll make use of MDAnalysis' functionality to query the atoms of interest: the protein (minus hydrogens) and the ligand (residue OSE):

In [10]:
from nanover.mdanalysis import frame_data_to_mdanalysis

universe = frame_data_to_mdanalysis(utilities.current_frame)

PROTEIN_ATOMS = universe.select_atoms("protein and not type H*")
LIGAND_ATOMS = universe.select_atoms("resname OSE")

We'll colour and render the protein with a spline, or ribbon, renderer.
Some things you can try: 
* Change the render: `spline`, `geometric spline`. Or comment out the `sequence` line and try `liquorice`,`noodles`, `cycles`, `ball and stick`.
* Change the color: set it to be one color, or try some different matplotlib [color maps](https://matplotlib.org/3.1.0/tutorials/colors/colormaps.html), e.g. `rainbow` or `magma`.
* Change the scale.

In [11]:
utilities.selections.update_selection(
    "protein",
    particle_ids=PROTEIN_ATOMS.indices,
    renderer = {
        'sequence': 'polypeptide',
        'color': {
            'type': 'residue index in entity',
            'gradient': get_matplotlib_gradient('rainbow')
        },
        'render': 'geometric spline',
        'scale': 0.2
    },
    interaction_method="single",
)

Let's reintroduce the ligand, oseltamivir, and make it so we interact with it as a group.

In [12]:
utilities.selections.update_selection(
    "ligand",
    particle_ids=LIGAND_ATOMS.indices,
    renderer = {
        'color': 'cpk',
        'scale': 0.1,
        'render': 'liquorice'
    },
    velocity_reset=True,
    interaction_method="group",
)

If you've done all that, you'll have something that looks like this:

![Neuraminidase Geometric](../figures/neuraminidase_geometric_spline.png)